# Klasifikasi Motif Batik Indonesia
Berdasarkan Pembelajaran Representasi Citra (*Image Representation Learning*) menggunakan PyTorch.

Notebook ini menunjukkan cara meload dataset, visualisasi augmentasi, dan proses training menggunakan `resnet18`.

In [ ]:
import os
import sys
from pathlib import Path
import random

import torch
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms.functional as TF

# Tambahkan path src agar bisa import modul custom kita
ROOT_DIR = Path(os.getcwd())
sys.path.insert(0, str(ROOT_DIR))

from src.dataset import BatikDataset, get_dataloaders
from src.model import get_model, count_parameters
from src.utils import get_default_transforms, IMAGENET_MEAN, IMAGENET_STD

## 1. Setup Dataset & Dataloaders
Kita gunakan dataset yang sudah displit di `data/processed`.

In [ ]:
DATA_DIR = ROOT_DIR / "data/processed"
BATCH_SIZE = 16

train_tf = get_default_transforms(train=True)
val_tf = get_default_transforms(train=False)

train_loader, val_loader, class_names = get_dataloaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=2,
    train_transform=train_tf,
    val_transform=val_tf
)

print(f"Total kelas: {len(class_names)}")
print(f"Sampel Train: {len(train_loader.dataset)}")
print(f"Sampel Val: {len(val_loader.dataset)}")
print("Daftar motif:", class_names)

## 2. Visualisasi Batch Data
Menampilkan citra dari dataloader untuk memastikan augmentasi berjalan (RandomCrop, Flip, Jitter) dan label sudah sesuai.

In [ ]:
def unnormalize(tensor_img):
    """Kembalikan tensor ke range 0-1 untuk visualisasi."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = tensor_img * std + mean
    return img.clamp(0, 1)

# Ambil satu batch dari train_loader
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = unnormalize(images[i])
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.set_title(class_names[labels[i].item()])
        ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Inisialisasi Model ResNet18
Kita gunakan pre-trained ResNet18 yang lapisan akhir klasifikasinya disesuaikan dengan jumlah motif batik kita.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

model = get_model(
    model_name="resnet18", 
    num_classes=len(class_names),
    pretrained=True,
    device=device
)

print(f"Total parameter model: {count_parameters(model):,}")

## 4. Setup Training Loop (Contoh Mini)
Untuk training komprehensif, gunakan script `src/train.py` lewat terminal. Di notebook ini kita contohkan loop untuk 1 epoch.

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Set model ke mode training
model.train()
running_loss = 0.0
all_preds, all_labels = [], []

print("Memulai iterasi epoch...")
for batch_idx, (imgs, lbls) in enumerate(train_loader):
    imgs, lbls = imgs.to(device), lbls.to(device)
    
    optimizer.zero_grad()
    outputs = model(imgs)
    loss = criterion(outputs, lbls)
    loss.backward()
    optimizer.step()
    
    running_loss += loss.item() * imgs.size(0)
    preds = outputs.argmax(dim=1).detach().cpu().numpy()
    all_preds.extend(preds)
    all_labels.extend(lbls.cpu().numpy())
    
    if (batch_idx + 1) % 10 == 0:
        print(f"Batch {batch_idx + 1}/{len(train_loader)} - Loss sementara: {loss.item():.4f}")

epoch_loss = running_loss / len(train_loader.dataset)
epoch_acc = accuracy_score(all_labels, all_preds)
print(f"\nHasil Akhir Epoch - Loss: {epoch_loss:.4f} | Akurasi: {epoch_acc:.4f}")